# LCEL: Runnable을 연결하는 데이터 흐름

LCEL(LangChain Expression Language)은 여러 Runnable을 순서 또는 분기로 연결하는 LangChain의 표현 방식이다. 프롬프트, 모델, 파서, Retriever처럼 역할이 다른 구성 요소도 Runnable로 준비하면 같은 방식으로 조합할 수 있다.

```python
chain = first | second | third
```

`|`는 앞 Runnable의 출력을 뒤 Runnable의 입력으로 전달한다. 이렇게 연결해 만든 전체 처리 경로를 Chain이라고 한다. 따라서 LCEL에서는 각 단계의 **입력 → 변환 → 출력 → 다음 사용처**를 확인하는 것이 중요하다.

## 핵심 용어

- `Runnable`: 입력을 받아 작업을 실행하고 결과를 반환하는 공통 실행 단위이다.
- `|`: 두 Runnable을 앞뒤 순서로 연결하는 LCEL 연산자이다.
- `Chain`: 하나 이상의 Runnable을 연결해 완성한 전체 처리 경로이다.


## Runnable의 세 가지 실행 방법

`Runnable`은 입력을 받아 정해진 작업을 실행하고 결과를 반환하는 LangChain의 공통 실행 단위이다. 구현체가 달라도 같은 실행 메서드를 사용할 수 있다.

### 메서드 선택 기준

- `invoke(input)`: 입력 하나를 처리하고 결과 하나를 반환한다.
- `batch([input1, input2])`: 여러 입력에 같은 작업을 적용하고 입력 순서에 대응하는 결과 목록을 반환한다.
- `stream(input)`: 출력을 iterator로 반환한다. 구현체가 여러 chunk를 만들면 조각별로 전달하고, 기본 구현은 최종 결과 하나만 반환할 수 있다.

### stream()이 반환하는 iterator

`stream()`의 반환값은 `iterator`이다.

- `next()`를 호출할 때 다음 값 하나를 반환한다.
- 마지막으로 읽은 위치를 기억한다.
- 반복문이나 `list()`에 전달하면 남아 있는 값을 끝까지 소비한다.

이어지는 코드는 먼저 `invoke()`와 `batch()`를 확인하고, 이후 generator와 `stream()`으로 출력 조각을 처리한다.


## Runnable 조합 요소와 선택 기준
 
Runnable은 한 가지 클래스가 아니라 같은 실행 방식을 따르는 여러 구성 요소의 공통 형태이다. 필요한 데이터 흐름에 따라 알맞은 구현체를 선택한다.

### 구성 요소의 역할

- `RunnableLambda`: 완성된 값 하나를 일반 Python 함수나 호출 가능한 객체로 변환한다.
- `RunnableGenerator`: 앞 단계가 보내는 출력 조각을 받는 즉시 새로운 출력 조각으로 변환한다.
- `RunnableSequence`: 앞 단계의 출력을 다음 단계의 입력으로 전달한다.
- `RunnableParallel`: 같은 입력을 여러 Runnable에 전달하고 결과를 이름 있는 dict로 묶는다.
- `RunnablePassthrough`: 입력을 변경하지 않고 전달해 원본과 파생 결과를 함께 사용하게 한다.

### 연결 전에 확인할 것

- Sequence는 앞 단계의 **출력 자료형**과 다음 단계의 **입력 자료형**이 맞아야 한다.
- Parallel은 모든 분기가 동일한 원본 입력을 처리할 수 있어야 한다.
- 원본 입력이 뒤에서 다시 필요하면 Passthrough로 보존한다.


## RunnableLambda로 문자열 길이 계산

`callable`은 `function(...)`처럼 괄호를 붙여 호출할 수 있는 함수나 객체이다. lambda 함수와 `def`로 만든 함수가 대표적인 callable이다.

### RunnableLambda를 사용하는 이유

일반 Python 함수는 그대로 호출할 수 있지만 `invoke()`, `batch()`, `stream()`과 LCEL 연결 기능은 제공하지 않는다. `RunnableLambda`는 함수의 계산 규칙을 유지하면서 이러한 Runnable 인터페이스를 추가한다.

이어지는 코드는 문자열을 문자 수 정수로 바꾸는 lambda 함수를 감싸고 `invoke()`로 입력 하나를 처리한다.


In [2]:
from langchain_core.runnables import RunnableLambda, RunnableGenerator

# 입력된 문자열의 길이를 반환하는 RunnableLambda 작성
length_runnable = RunnableLambda(lambda text: len(text))

input_text = '안녕하세요~😘'
char_count = length_runnable.invoke(input_text)
print({'input': input_text, 'char_count': char_count})

{'input': '안녕하세요~😘', 'char_count': 7}


## 이름 있는 Python 함수를 Runnable으로 감싸기

`RunnableLambda`에는 lambda 함수뿐 아니라 `def`로 정의한 함수도 전달할 수 있다.

- lambda 함수: 한 줄로 표현할 수 있는 짧은 변환에 적합하다.
- 이름 있는 함수: 타입 힌트, 여러 처리 단계, 재사용이 필요한 변환에 적합하다.

함수를 감싼 뒤에는 두 방식 모두 `invoke()`, `batch()`, `stream()`을 같은 형태로 사용한다. 이어지는 코드는 `count_characters()`를 감싸 다음 batch 예제에서도 재사용한다.


In [3]:
from langchain_core.runnables import RunnableLambda

def count_characters(text: str) -> int:
    return len(text)

# 입력된 문자열의 길이를 반환하는 RunnableLambda 작성
named_length_runnable = RunnableLambda(count_characters)

named_text = '홍길동~😘'
named_count = named_length_runnable.invoke(named_text)
print({'input': named_text, 'char_count': named_count})

{'input': '홍길동~😘', 'char_count': 5}


## batch는 입력 순서를 보존해 여러 요청을 처리한다

`batch()`는 같은 Runnable에 여러 입력을 전달할 때 사용하는 실행 메서드이다. 입력 목록의 각 원소에 같은 작업을 적용하고 결과를 목록으로 반환한다.

### 입력과 출력의 대응

- 입력: `[input1, input2, input3]` 형태의 목록이다.
- 처리: 각 입력을 같은 Runnable에 하나씩 전달한다.
- 출력: `[output1, output2, output3]` 형태의 목록이다.
- 순서: `output1`은 `input1`, `output2`는 `input2`의 결과이다.

따라서 여러 요청을 함께 처리해도 입력과 결과를 같은 인덱스로 연결할 수 있다.

### 주의할 점

`batch()` 한 번의 호출과 외부 API 한 번의 요청은 항상 일치하지 않는다.

- Runnable 구현체에 따라 여러 `invoke()`를 병렬로 실행할 수 있다.
- 공급자가 전용 batch 기능을 제공하면 이를 사용할 수도 있다.
- 여러 입력을 다루는 코드는 단순해지지만 API 요청 횟수 감소를 보장하지 않는다.


In [4]:
batch_inputs = ['안녕하세요?', '뉘슈?', '누구세요?']

named_length_runnable = RunnableLambda(count_characters)

# batch를 이용해서 여러 값을 목록 형태로 전달
batch_counts = named_length_runnable.batch(batch_inputs)

print({'input': batch_inputs, 'char_count': batch_counts})

{'input': ['안녕하세요?', '뉘슈?', '누구세요?'], 'char_count': [6, 3, 5]}


## 수치 변환도 같은 batch 입력·출력 규칙을 사용한다

Runnable의 입력 자료형은 문자열로 제한되지 않는다. callable이 숫자를 받을 수 있다면 `invoke()`와 `batch()`도 숫자를 그대로 처리한다.

이어지는 코드는 섭씨 값 하나를 화씨 값 하나로 바꾸는 공식을 Runnable로 만들고, 다섯 측정값에 같은 계산을 적용한다. 결과 목록의 각 위치는 입력 목록의 같은 위치에 대응한다.


In [5]:
# 섭씨 -> 화씨 변경
celsius_to_fahrenheit = RunnableLambda(
    lambda celsius: (celsius * 9/5) + 32
)

celsius_values = [-10, 0, 20, 39, 100]
fahrenheit_values = celsius_to_fahrenheit.batch(celsius_values)
print({
    'celsius': celsius_values,
    'fahrenheit': fahrenheit_values
})

{'celsius': [-10, 0, 20, 39, 100], 'fahrenheit': [14.0, 32.0, 68.0, 102.2, 212.0]}


### 확인 문제 1: 할인 가격을 invoke와 batch로 계산하기

앞의 예제에서는 하나의 RunnableLambda를 `invoke()`와 `batch()`에서 같은 방식으로 사용했다. 이제 가격 하나와 가격 목록에 10% 할인 규칙을 적용한다.

- 입력: 정수 가격 하나 또는 정수 가격 목록이다.
- 변환: 입력 가격에 `0.9`를 곱하고 정수로 반환한다.
- 목표: 단일 입력과 목록 입력에 같은 할인 규칙을 적용한다.

먼저 두 출력값을 예상한 뒤 코드를 실행해 비교한다.


In [7]:
discount_price = RunnableLambda(lambda price: int(price * 0.9))

single_discount = discount_price.invoke(15000)
batch_discount = discount_price.batch([10000, 13000, 47500])
print({'single': single_discount, 'batch': batch_discount})

{'single': 13500, 'batch': [9000, 11700, 42750]}


## generator는 값을 한 번씩 소비한다

`generator`는 필요한 시점에 값을 `yield`를 이용해 하나씩 만드는 iterator 객체이다. 모든 값을 미리 만들지 않으므로 큰 데이터나 stream을 순서대로 처리할 때 유용하다.

### yield의 역할

`yield`는 현재 값을 호출자에게 내보내고 함수 실행을 잠시 멈춘다.

- `return`: 값을 반환하고 함수 실행을 끝낸다.
- `yield`: 값을 하나 내보내고 실행 위치와 지역 변수 상태를 기억한다.
- 다음 `next()`가 호출되면 멈췄던 `yield`의 다음 줄부터 실행한다.
- 함수의 실행이 끝나면 generator도 종료된다.


In [23]:
def generate_numbers(values):
    for value in values:
        yield value

number_stream = generate_numbers(range(10))

print(type(number_stream).__name__)

generator


## next로 첫 값 하나를 소비하기

next(number_stream)는 generator의 현재 위치에서 다음 값 하나를 꺼내고 위치를 한 칸 이동시킨다. 여기서 0을 소비했으므로 다음 셀에서 같은 generator를 반복하면 0은 다시 나오지 않는다.


In [24]:
first_number = next(number_stream)
print("first_number:", first_number) # 처음 0

first_number: 0


## 같은 generator에 남은 값 확인하기

앞 셀에서 0을 소비했으므로 `list(number_stream)`은 1부터 9까지 남아 있는 값만 새 목록으로 모은다. 목록으로 변환한 뒤에는 generator도 끝까지 소비되므로, 같은 객체를 다시 반복하는 대신 필요하면 새 generator를 만들어야 한다.


In [25]:
remaining_numbers = list(number_stream)
print("remaining_numbers:", remaining_numbers)

remaining_numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9]


### 확인 문제 2: generator의 남은 값 예측하기

`generate_numbers(range(5))`로 새로운 generator를 만든다. `next()`를 두 번 호출한 뒤 같은 객체를 `list()`로 변환할 때 각 변수에 어떤 값이 저장되는지 먼저 예상한다.

- 첫 번째 `next()`: 현재 위치의 첫 값을 소비한다.
- 두 번째 `next()`: 다음 위치의 값을 소비한다.
- `list()`: 이미 소비한 값을 제외하고 남은 값을 끝까지 모은다.

예상한 결과와 실제 `first`, `second`, `remaining`을 비교한다.


In [26]:
practice_numbers = generate_numbers(range(5))

first = next(practice_numbers)
second = next(practice_numbers)
remaining = list(practice_numbers)

print({
    'first': first,
    'second': second,
    'remaining': remaining
})

{'first': 0, 'second': 1, 'remaining': [2, 3, 4]}


## RunnableLambda와 RunnableGenerator의 stream 차이

`chunk`는 `stream()`이 한 번에 전달하는 출력 조각이다. Python의 특정 자료형 이름은 아니며, 텍스트 한 글자·토큰 일부·메시지 블록처럼 Runnable이 나누어 보내는 단위를 뜻한다.

### 두 Runnable이 chunk를 만드는 방식

- `RunnableLambda`: 완성된 입력을 callable에 전달하고 기본적으로 최종 결과 하나를 chunk로 내보낸다.
- generator 함수를 감싼 `RunnableLambda`: 완성된 입력을 받은 뒤 함수가 `yield`하는 값을 여러 chunk로 내보낸다.
- `RunnableGenerator`: 앞 단계의 입력 chunk iterator를 받아 각 chunk를 처리하는 즉시 새 chunk를 내보낸다.

### 선택 기준

- 완성된 값 하나를 변환하면 `RunnableLambda`를 사용한다.
- 앞 단계의 chunk 경계를 유지하면서 즉시 변환해야 하면 `RunnableGenerator`를 사용한다.

이어지는 코드는 최종 문자열 하나, 문자 단위 chunk, 가공된 문자 chunk를 비교한다. 생성자의 `transform` 인자와 스트리밍 방식은 [RunnableGenerator 공식 참조](https://reference.langchain.com/python/langchain-core/runnables/base/RunnableGenerator)에서 확인할 수 있다.


In [28]:

stream_input = 'Hi 안녕🦑'

# 완성된 str 하나를 대문자로 바꾼 후 str 하나로 반환한다.
plain_uppercase = RunnableLambda(lambda text: text.upper())

# stream()이 반환한 iterator()를  list()가 끝까지 소비해서
# 실제 chunk 목록을 만든다
lambda_chunks = list(plain_uppercase.stream(stream_input))
print("RunnableLambda chunks:", lambda_chunks)
print(len(lambda_chunks))

RunnableLambda chunks: ['HI 안녕🦑']
1


### generator를 반환하는 RunnableLambda

`RunnableLambda`가 감싼 callable 자체가 generator 함수이면 최종 값 하나 대신 여러 값을 `yield`할 수 있다.

- 입력: 완성된 문자열 하나이다.
- 변환: 문자열을 한 글자씩 순회한다.
- 출력: `Iterator[str]` 형태의 문자 chunk이다.

입력 전체가 준비된 뒤 chunk를 만들기 때문에, 앞 단계의 chunk를 즉시 처리하는 `RunnableGenerator`와는 역할이 다르다.


In [29]:
# abc == ABstract Class (추상 클래스)
# - 여기서는 상위 타입의 변수(다형성)로 사용
from collections.abc import Iterator

# 전달 받은 완성된 문자 text를
# 문자 하나씩 yield하는 generator 함수를 정의
# -> generator는 Iterator의 자식
# -> stream()에 의해서 문자(chunk) 하나씩 순차적으로 반환
# ==  Iterator[str]
def generate_characters(text: str) -> Iterator[str]:
    for char in text:
        yield char

# 위 함수를 RunnableLambda가 사용할 callable(함수)로 등록
character_source = RunnableLambda(generate_characters)

# Runnable.stream() : 결과를 Stream 형태(chunk 단위)로 받음
# == chunk 단위로 반복해서 출력하는 Iterator의 형태

# 입력 : 'HI 안녕🦑'
# 출력 : Iterator ('H', 'I', ' ', '안', '녕', '🦑')
# list 형변환: list ['H', 'I', ' ', '안', '녕', '🦑']
source_chunks = list(character_source.stream(stream_input))

print(source_chunks)


['H', 'i', ' ', '안', '녕', '🦑']


### RunnableGenerator로 upstream chunk를 즉시 변환하기

`RunnableGenerator`는 앞 Runnable이 보내는 chunk를 모아서 기다리지 않고 도착하는 순서대로 변환할 때 사용한다.

### 사용 목적

- LLM의 스트리밍 응답을 chunk 단위로 꾸미거나 필터링한다.
- 전체 응답이 끝날 때까지 기다리지 않고 변환한 chunk를 다음 단계로 보낸다.
- 중간 변환을 추가해도 앞 단계의 chunk 경계와 스트리밍 흐름을 유지한다.

### 입력과 출력

- `transform` 입력: `Iterator[입력 chunk]`이다.
- `transform` 출력: `Iterator[출력 chunk]`이다.


In [32]:

from langchain_core.runnables import RunnableGenerator

# 문자 chunk generator(== Iterator)를 입력 받아
# 각 chunk가 가공된 generator를 반환
def wrap_chunks(chunks: Iterator[str]) -> Iterator[str]:
    for chunk in chunks:
        yield '[' + chunk + ']'  # '[A]'

chunk_wrapper = RunnableGenerator(wrap_chunks)

# Runnable이란? 입력 -> 작업실행 -> 출력  작업단위
# - Runnable 끼리는 이어서 실행 가능 (이전 출력 -> 다음 입력)
chunk_preserving_chain = character_source | chunk_wrapper

# 입력: 'Hi 안녕🦑'
# character_source: Iterator ['H', 'i', ' ', '안', '녕', '🦑']
# chunk_wrapper: Iterator ['[H]', '[i]', '[ ]', '[안]', '[녕]', '[🦑]']

wrapped_chunks = [] # chunk 누적

# 반복문을 이용해 stream()이 chunk를 보낼 때 마다 출력 + 목록 누적
for wrap_chunk in chunk_preserving_chain.stream(stream_input):
    print(wrap_chunk, end='', flush=True)
    wrapped_chunks.append(wrap_chunk)

print()
print('collected chunks:', wrapped_chunks)

[H][i][ ][안][녕][🦑]
collected chunks: ['[H]', '[i]', '[ ]', '[안]', '[녕]', '[🦑]']


### 확인 문제 3: 공백을 제외한 문자 chunk 만들기

`character_source`가 만드는 문자 chunk를 새로운 `RunnableGenerator`로 받는다. 공백 chunk는 전달하지 않고 나머지 문자는 `<문자>` 형식으로 바꾼다.

- 입력 문자열: `'A B'`이다.
- 첫 단계 출력: `'A'`, `' '`, `'B'` 문자 chunk이다.
- 두 번째 단계 출력: 공백을 제외하고 꺾쇠로 감싼 문자 chunk이다.

완성 문자열을 한 번에 가공하지 않고 upstream chunk를 하나씩 판단하는 흐름에 집중한다.


In [33]:

# 공백 필터링 Generator
def wrap_non_space_chunks(chunks: Iterator[str]) -> Iterator[str]:
    for chunk in chunks:

        # 만약 chunk 값이 ' '(띄어쓰기 한 칸)인 경우 다음 반복으로 넘어가기
        if chunk == ' ':
            continue

        yield '<' + chunk + '>'

# RunnableGenerator로 변환
non_space_wrapper = RunnableGenerator(wrap_non_space_chunks)

# 입력 값을 문자 하나 단위의 chunk로 만들어서
# 띄어쓰기 제거 + <chunk> 감싸는 Generator 반환하는 체인
non_space_chain = character_source | non_space_wrapper

non_space_chunks = list(non_space_chain.stream(stream_input))

print(non_space_chunks)

['<H>', '<i>', '<안>', '<녕>', '<🦑>']


## RunnableSequence: 앞 단계 출력을 다음 단계 입력으로 전달하기

`RunnableSequence`는 여러 Runnable을 순서대로 실행하는 연결 구조이다. 앞 단계의 출력이 다음 단계의 입력이 되므로 각 단계의 자료형이 서로 맞아야 한다.

### 작성 방법

- 생성자: `RunnableSequence(first, second)`처럼 실행 순서를 인자로 전달한다.
- LCEL: `first | second`처럼 왼쪽에서 오른쪽으로 연결한다.

두 방법은 같은 Sequence를 만든다. 여러 단계를 읽기 쉽게 연결할 때는 데이터 흐름이 보이는 `|` 표현을 주로 사용한다.

이어지는 코드는 생성자와 `|`로 만든 Sequence가 같은 결과를 반환하는지 비교한다.


In [34]:
from langchain_core.runnables import RunnableSequence

# {'foo': value} dict 형태로 반환
wrap_foo = RunnableLambda(lambda value: {'foo': value})

# 입력 값을 3번 반복하여 list 3개로 반환
repeat_mapping = RunnableLambda(lambda item: [item] * 3)

# 1) LCEL 구문 (|) 이용
sequence_chain = wrap_foo | repeat_mapping

# 2) RunnableSequence 이용
sequence_by_constructor = RunnableSequence(
    wrap_foo, repeat_mapping
)

# 확인
chain_result = sequence_chain.invoke(77)
constructor_result = sequence_by_constructor.invoke(77)

print("chain_result:", chain_result)
print("constructor_result:", constructor_result)


chain_result: [{'foo': 77}, {'foo': 77}, {'foo': 77}]
constructor_result: [{'foo': 77}, {'foo': 77}, {'foo': 77}]


## 순서를 바꾸면 자료형 흐름도 바뀐다

Sequence는 같은 Runnable을 사용해도 연결 순서에 따라 중간값과 최종 결과가 달라진다. 각 단계가 직전 단계의 출력 전체를 입력으로 받기 때문이다.

이어지는 코드는 앞의 연결을 유지한 채 순서만 뒤집은 `reverse_sequence`를 별도로 만들고 두 결과 구조의 차이를 확인한다.


In [36]:
reverse_sequence = repeat_mapping | wrap_foo
reverse_result = reverse_sequence.invoke(33)
print(reverse_result)

{'foo': [33, 33, 33]}


## 문자열 정리와 Unicode 코드 포인트 변환

두 문자열 변환을 연결해 Sequence의 자료형 흐름을 확인한다.

- 첫 단계: 문자열에서 일반 공백을 제거하고 새 문자열을 반환한다.
- 두 번째 단계: 각 문자를 `ord()`에 전달해 Unicode 코드 포인트 목록을 반환한다.
- 연결 조건: 첫 단계의 `str` 출력이 두 번째 단계의 `str` 입력과 일치한다.

`ord()`는 ASCII 전용 함수가 아니므로 영문, 한글, 이모지를 모두 숫자로 바꿀 수 있다. 다만 이 숫자는 문자에 부여된 코드 포인트이며 언어 모델의 토큰 ID나 임베딩 벡터와는 다르다.


In [37]:
remove_spaces = RunnableLambda(
    lambda text: text.replace(' ', '')
)

to_code_points = RunnableLambda(
    lambda text: [ord(char) for char in text]
)

# 공백 제거 -> 유니 코드 변환 체인
text_to_code_points: RunnableSequence = remove_spaces | to_code_points

english_code_points = text_to_code_points.invoke('a b  c d    ')
print(english_code_points)

[97, 98, 99, 100]


## Sequence의 첫 변환만 독립적으로 확인하기

여러 Runnable을 연결한 Chain에서 결과가 예상과 다르면 각 단계를 따로 실행해 중간값을 확인한다. 이를 통해 어느 단계에서 값이나 자료형이 달라졌는지 범위를 좁힐 수 있다.

이어지는 코드는 첫 Runnable인 `remove_spaces`만 실행해 공백 제거 결과를 확인한다.


In [38]:
space_removed_text = remove_spaces.invoke('한  글 도   되 나 ?')
print(space_removed_text)

한글도되나?


## Unicode 코드 포인트 변환을 독립적으로 확인하기

두 번째 Runnable인 `to_code_points`는 문자열 하나를 받아 각 문자의 Unicode 코드 포인트가 담긴 목록을 반환한다.

- 입력: `str` 하나이다.
- 변환: 문자열을 순회하며 각 문자에 `ord()`를 적용한다.
- 출력: 입력 문자와 같은 순서를 유지하는 `list[int]`이다.

이어지는 코드는 한글 두 글자를 입력해 각 문자와 숫자의 대응을 확인한다.


In [39]:
korean_code_points = to_code_points.invoke('한글')
print(korean_code_points)

[54620, 44544]


## 완성한 텍스트 Sequence를 한글 입력에 적용하기

앞에서는 두 Runnable을 각각 실행해 중간 결과를 확인했다. 이제 두 단계를 연결한 `text_to_code_points`에 한글 문장 하나를 전달한다.

최종 출력은 공백 제거 문자열이 아니라 두 번째 Runnable이 만든 `list[int]`이다. 이어지는 코드는 원문과 최종 숫자 목록을 함께 출력해 입력·출력 자료형을 비교한다.


In [40]:
sequence_input = "한   글   도 되나?"

sequence_code_points = text_to_code_points.invoke(sequence_input)

print({
    'input':sequence_input,
    'code_points':sequence_code_points
})

{'input': '한   글   도 되나?', 'code_points': [54620, 44544, 46020, 46104, 45208, 63]}


### 확인 문제 4: 세 Runnable으로 주문 코드를 정규화하기

주문 코드 `'  skn-33  '`를 `str → str → str → dict` 순서로 변환한다.

1. 앞뒤 공백을 제거한다.
2. 알파벳을 대문자로 바꾼다.
3. 정규화된 코드와 하이픈 앞의 prefix를 dict로 만든다.
4. 세 Runnable을 `|`로 연결해 `order_code_chain`을 만든다.
5. 같은 chain을 `invoke()`와 `batch()`에서 실행한다.

먼저 각 단계의 출력 자료형을 적은 뒤 직접 chain을 구성하고 결과를 확인한다.


In [44]:
# 입력(str) → 공백제거(str) → 대문자(str) → dict

# 앞뒤 공백 제거
strip_order_code = RunnableLambda(lambda code: code.strip())

# 대문자
uppercase_order_code = RunnableLambda(lambda code: code.upper())

# 정규화된 코드를 보존하고 접두사(prefix) 분리한 dict
to_order_mapping = RunnableLambda(lambda code: {
    "order_code": code,
    "prefix": code.split("-")[0]
})

order_code_chain = (
        strip_order_code |
        uppercase_order_code |
        to_order_mapping
)

normalized_order = order_code_chain.invoke('  skn-42  ')

normalized_orders = order_code_chain.batch([
    '  skn-7  ',
    '  lab-12  ',
])

print({
    'single': normalized_order,
    'batch': normalized_orders
})

# 'batch': [{'order_code': 'SKN-7', 'prefix': 'SKN'}, {'order_code': 'LAB-12', 'prefix': 'LAB'}]}

{'single': {'order_code': 'SKN-42', 'prefix': 'SKN'}, 'batch': [{'order_code': 'SKN-7', 'prefix': 'SKN'}, {'order_code': 'LAB-12', 'prefix': 'LAB'}]}


## RunnableParallel: 같은 입력을 이름 있는 여러 분기로 전달하기

`RunnableParallel`은 입력 하나를 여러 Runnable에 동시에 전달하고 각 결과를 하나의 dict로 묶는 분기 구조이다.

### 입력과 출력

- 입력: 모든 분기가 함께 받을 원본 값 하나이다.
- 처리: 각 분기가 서로 독립적으로 같은 입력을 변환한다.
- 출력: 생성자에 작성한 keyword가 key가 되고 각 분기의 반환값이 value가 되는 dict이다.

### Sequence와의 차이

- Sequence: 앞 단계의 출력이 다음 단계의 입력이 된다.
- Parallel: 각 분기가 같은 원본 입력을 받으며 다른 분기의 결과를 직접 읽지 않는다.

I/O 대기 시간이 있는 작업은 병렬 처리의 이점을 얻을 수 있다. 이어지는 작은 계산 예제는 속도보다 분기 입력과 출력 dict 구조를 확인하는 데 목적이 있다.


In [46]:
# RunnableParallel: 여러 Runnable에 동시에 값을 전달하고
# 하나의 dict로 묶어서 결과를 반환

from langchain_core.runnables import RunnableParallel

# RunnableParallel(key=Runnable, ...)
parallel_chain = RunnableParallel(
    result1 = wrap_foo,
    result2 = repeat_mapping
)

parallel_result = parallel_chain.invoke(23)
print(parallel_result)

{'result1': {'foo': 23}, 'result2': [23, 23, 23]}


## RunnablePassthrough로 원본 입력과 파생 결과를 함께 보관하기

`RunnablePassthrough`는 받은 입력을 변경하지 않고 그대로 반환하는 Runnable이다.

### 사용하는 이유

Parallel의 다른 분기에서 파생 값을 만들더라도 이후 단계가 원본 입력을 다시 필요로 할 수 있다. 이때 Passthrough 분기를 추가하면 원본과 파생 결과를 하나의 dict에 함께 보관할 수 있다.

### 작성 방법

- `RunnablePassthrough()`: 입력을 그대로 반환한다.
- `{'original': RunnablePassthrough(), 'derived': some_runnable}`: 같은 입력에서 원본과 파생 값을 만든다.
- LCEL에서 Runnable을 값으로 가진 dict는 `RunnableParallel`로 변환된다.

RAG에서는 질문 원문을 보존하면서 다른 분기에서 검색 문맥을 만드는 구조에 활용할 수 있다.


In [47]:
from langchain_core.runnables import RunnablePassthrough

pass_branch_chain = RunnableParallel(
    # original 분기는 입력 값을 변경하지 않고 그대로 반환
    original = RunnablePassthrough(),

    result1 = wrap_foo,
    result2 = repeat_mapping
)

pass_result = pass_branch_chain.invoke(3)
print(pass_result)

{'original': 3, 'result1': {'foo': 3}, 'result2': [3, 3, 3]}


## 여러 수치 계산을 RunnableParallel로 묶기

같은 입력에서 서로 의존하지 않는 계산은 Parallel의 독립 분기로 구성할 수 있다.

정수 하나를 제곱, 팩토리얼, 짝수 여부를 계산하는 세 Runnable에 전달하도록 구성한다. 각 반환값은 `square`, `factorial`, `is_even` key로 구분하므로 다음 단계가 필요한 결과를 이름으로 선택할 수 있다.


In [48]:
import math

# 입력 값 제곱
square_runnable = RunnableLambda(lambda value: value ** 2)

# 1부터 지정된 자연수의 곱
factorial_runnable = RunnableLambda(math.factorial)

# 짝수 판별
is_even_runnable = RunnableLambda(lambda value: value % 2 == 0)


metrics_chain = RunnableParallel(
    square = square_runnable,
    factorial = factorial_runnable,
    is_even = is_even_runnable
)

metrics_result = metrics_chain.invoke(8)
print(metrics_result)

{'square': 64, 'factorial': 40320, 'is_even': True}


## 직접 적용하기: 원본 보존과 병렬 계산을 순차 연결하기

### 주문 원본과 파생 값을 병렬로 만들기

이번에는 Passthrough, Parallel, Sequence를 하나의 주문 처리 흐름에 함께 사용한다. 첫 단계에서는 주문 원본을 보존하면서 다음 값을 독립적으로 계산한다.

- `original`: 입력 주문 dict 전체이다.
- `subtotal`: 단가와 수량을 곱한 소계이다.
- `shipping_fee`: 회원 여부에 따른 배송비이다.
- `is_member`: 다음 요약 단계에서 사용할 회원 여부이다.

Parallel 분기는 서로의 결과를 읽지 않으므로 필요한 값은 각자 원본 주문에서 가져온다.


In [50]:
order_input = {
    'unit_price': 12000,
    'quantity': 3,
    'member': True
}

feature_chain = RunnableParallel(
    original = RunnablePassthrough(), #원본 보존

    # 단가 * 수량
    subtotal = RunnableLambda(
        lambda order: order['unit_price'] * order['quantity']
    ),

    # 회원이면 0원, 비회원은 3000원
    shipping_fee = RunnableLambda(
        lambda order: 0 if order['member'] else 3000
    ),
    is_member = RunnableLambda(
        lambda order: order['member']
    )
)

# 병렬처리 확인
feature_result = feature_chain.invoke(order_input)
print(feature_result)

{'original': {'unit_price': 12000, 'quantity': 3, 'member': True}, 'subtotal': 36000, 'shipping_fee': 0, 'is_member': True}


### 병렬 결과로 최종 결제 정보 만들기

Parallel이 만든 특징 dict를 다음 `RunnableLambda`의 입력으로 연결한다.

- 입력: 원본 주문과 소계·배송비·회원 여부가 담긴 특징 dict이다.
- 변환: 소계와 배송비를 더하고 회원 여부에 맞는 요약 문장을 만든다.
- 출력: 기존 key에 `final_payment`와 `summary`가 추가된 최종 dict이다.

두 Runnable을 `|`로 연결해 `주문 dict → 특징 dict → 최종 결제 dict`를 처리하는 전체 Chain을 만든다.


In [51]:
def build_payment_summary(feature:dict) -> dict:

    # 최종 가격
    final_payment = feature['subtotal'] + feature['shipping_fee']

    # 회원 여부
    member_label = '회원' if feature['is_member'] else '비회원'

    return {
        **feature,
        'final_payment': final_payment,
        'summary':(
            f'{member_label} 주문: 소계 {feature["subtotal"]:,}원 + '
            f'배송비 {feature["shipping_fee"]:,}원 = {final_payment:,}원'
        )
    }

# 함수 -> Runnable로 변환 -> chain이 가능하게 변환
finalize_payment = RunnableLambda(build_payment_summary)

order_payment_chain: RunnableSequence = feature_chain | finalize_payment


payment_result = order_payment_chain.invoke(order_input)

print(payment_result)

{'original': {'unit_price': 12000, 'quantity': 3, 'member': True}, 'subtotal': 36000, 'shipping_fee': 0, 'is_member': True, 'final_payment': 36000, 'summary': '회원 주문: 소계 36,000원 + 배송비 0원 = 36,000원'}


### 스스로 적용할 문제

점수 목록 `[72, 88, 95, 61]`을 입력으로 받아 원본 목록, 평균, 최고점과 70점 이상 인원 수를 함께 보존하는 chain을 구성한다.

1. `RunnablePassthrough`와 `RunnableParallel`로 원본·평균·최고점·통과 인원을 key별로 만든다.
2. 다음 `RunnableLambda`에서 `pass_rate`를 추가한다.
3. 입력·중간·최종 자료형을 `list[int] → dict → dict`로 적는다.
4. 계산한 `pass_rate`가 입력 점수와 일치하는지 확인한다.

단순히 문자나 숫자 하나를 바꾸는 문제가 아니라, 각 분기가 같은 입력을 받고 다음 단계가 중간 dict의 key를 읽는 구조를 직접 설계한다.


In [52]:
score_input = [72, 88, 95, 61]

score_features = RunnableParallel(
    original=RunnablePassthrough(),
    average=RunnableLambda(lambda scores: sum(scores) / len(scores)),
    highest=RunnableLambda(max),
    passed_count=RunnableLambda(lambda scores: sum(score >= 70 for score in scores)),
)

def add_pass_rate(features: dict) -> dict:
    pass_rate = features['passed_count'] / len(features['original']) * 100
    return {**features, 'pass_rate': pass_rate}

score_summary_chain: RunnableSequence = (
    score_features | RunnableLambda(add_pass_rate)
)

score_result = score_summary_chain.invoke(score_input)
print(score_result)

{'original': [72, 88, 95, 61], 'average': 79.0, 'highest': 95, 'passed_count': 3, 'pass_rate': 75.0}


## 정리

LCEL은 Runnable을 순서와 분기로 연결하는 표현 방식이며, `|`는 앞 단계의 출력을 뒤 단계의 입력으로 연결하는 주요 연산자이다. 이 연결로 만든 전체 처리 경로가 Chain이다.

- `RunnableLambda`: 완성된 입력 하나를 Python callable로 변환한다.
- `RunnableGenerator`: upstream chunk를 받는 즉시 출력 chunk로 변환한다.
- `RunnableSequence`: 직전 단계의 출력을 다음 단계의 입력으로 전달한다.
- `RunnableParallel`: 같은 입력을 여러 분기에 전달하고 key가 있는 dict로 결과를 묶는다.
- `RunnablePassthrough`: 원본 입력을 변경 없이 유지해 파생 결과와 함께 사용하게 한다.

실무에서는 코드가 짧은지보다 앞 단계의 출력 자료형이 다음 단계의 입력과 맞는지, 분기의 key가 다음 사용처와 일치하는지를 먼저 확인한다.
